# Historical Trend Analysis for Store Performance

This notebook analyzes historical performance data to identify trends, forecast future performance, and help proactively adapt the retail mix.

## Objectives:
1. Identify temporal patterns (seasonality, trends) in store performance
2. Analyze performance trends by store category/type
3. Build forecasting models for future performance
4. Detect emerging vs declining store categories
5. Prepare insights and models for the Streamlit dashboard

## 1. Imports and Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical and time series libraries
from scipy import stats
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore")

# Set style
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# Import project constants
import constants.constants as cst
import constants.paths as pth

## 2. Data Loading and Preparation

In [ ]:
# Load dimension tables
dim_blocks = pd.read_csv(pth.DIM_BLOCKS, **cst.CSV_PARAMS)
dim_malls = pd.read_csv(pth.DIM_MALLS, **cst.CSV_PARAMS)

# Load fact tables
print("Loading fact_stores")
fact_stores = pd.read_csv(pth.FACT_STORES, **cst.CSV_PARAMS)
fact_malls = pd.read_csv(pth.FACT_MALLS, **cst.CSV_PARAMS)
fact_sri_scores = pd.read_csv(pth.FACT_SRI_SCORES, **cst.CSV_PARAMS)

# Load store financials
store_financials = pd.read_csv(pth.STORE_FINANCIALS, **cst.CSV_PARAMS)


print(f"fact_stores shape: {fact_stores.shape}")
print(f"store_financials shape: {store_financials.shape}")
print(f"dim_blocks shape: {dim_blocks.shape}")

In [ ]:
# Convert date columns to datetime
fact_stores["date"] = pd.to_datetime(fact_stores["date"], format="%d/%m/%Y")
fact_malls["date"] = pd.to_datetime(fact_malls["date"], format="%d/%m/%Y")

# Sort by date
fact_stores = fact_stores.sort_values("date")
fact_malls = fact_malls.sort_values("date")

stores_min, stores_max = fact_stores["date"].min(), fact_stores["date"].max()
malls_min, malls_max = fact_malls["date"].min(), fact_malls["date"].max()

print(f"Date range in fact_stores: {stores_min} to {stores_max}")
print(f"Date range in fact_malls: {malls_min} to {malls_max}")

In [ ]:
# Merge stores with block information to get category labels
stores_enriched = fact_stores.merge(
    dim_blocks[
        [
            "block_id",
            "store_code",
            "bl1_label",
            "bl2_label",
            "bl3_label",
            "gla",
            "gla_category",
        ]
    ],
    on="block_id",
    how="left",
    suffixes=("", "_dim"),
)

# Merge with mall information
stores_enriched = stores_enriched.merge(
    dim_malls[["id", "country", "mall_name"]],
    left_on="mall_id",
    right_on="id",
    how="left",
)

print(f"Enriched stores shape: {stores_enriched.shape}")
print("\nSample of enriched data:")
stores_enriched.head()

## 3. Exploratory Data Analysis - Time Patterns

In [ ]:
# Add temporal features
stores_enriched["year"] = stores_enriched["date"].dt.year
stores_enriched["month"] = stores_enriched["date"].dt.month
stores_enriched["week"] = stores_enriched["date"].dt.isocalendar().week
stores_enriched["day_of_week"] = stores_enriched["date"].dt.dayofweek
stores_enriched["day_name"] = stores_enriched["date"].dt.day_name()
stores_enriched["quarter"] = stores_enriched["date"].dt.quarter

# Create year-month for grouping
stores_enriched["year_month"] = stores_enriched["date"].dt.to_period("M")

print("Temporal features added.")

In [ ]:
# Overall traffic trends over time
daily_traffic = (
    stores_enriched.groupby("date")
    .agg(
        {
            "people_in": "sum",
            "people_window_flow": "sum",
            "store_average_dwell_time": "mean",
            "shopping_average_dwell_time": "mean",
        }
    )
    .reset_index()
)

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Total People In",
        "Window Flow",
        "Average Dwell Time (Store)",
        "Average Dwell Time (Shopping)",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1,
)

fig.add_trace(
    go.Scatter(
        x=daily_traffic["date"],
        y=daily_traffic["people_in"],
        mode="lines",
        name="People In",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_traffic["date"],
        y=daily_traffic["people_window_flow"],
        mode="lines",
        name="Window Flow",
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=daily_traffic["date"],
        y=daily_traffic["store_average_dwell_time"],
        mode="lines",
        name="Store Dwell",
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_traffic["date"],
        y=daily_traffic["shopping_average_dwell_time"],
        mode="lines",
        name="Shopping Dwell",
    ),
    row=2,
    col=2,
)

fig.update_layout(
    height=600, title_text="Overall Traffic Trends Over Time", showlegend=False
)
fig.show()

In [ ]:
# Monthly trends
monthly_traffic = (
    stores_enriched.groupby("year_month")
    .agg(
        {
            "people_in": "sum",
            "people_window_flow": "sum",
            "store_average_dwell_time": "mean",
            "store_code": "nunique",  # Number of active stores
        }
    )
    .reset_index()
)

monthly_traffic["year_month_str"] = monthly_traffic["year_month"].astype(str)

fig = px.line(
    monthly_traffic,
    x="year_month_str",
    y="people_in",
    title="Monthly Total Traffic Across All Stores",
    labels={"people_in": "Total People In", "year_month_str": "Month"},
)
fig.update_layout(height=400)
fig.show()

## 4. Category-Level Trend Analysis

In [ ]:
# Check distribution of categories
print("BL1 (High-level) Categories:")
print(stores_enriched["bl1_label"].value_counts())
print("\nBL2 (Mid-level) Categories:")
print(stores_enriched["bl2_label"].value_counts().head(20))

In [ ]:
# Trends by BL1 category over time
category_trends = (
    stores_enriched.groupby(["year_month", "bl1_label"])
    .agg(
        {
            "people_in": "sum",
            "store_average_dwell_time": "mean",
            "store_code": "nunique",
        }
    )
    .reset_index()
)

category_trends["year_month_str"] = category_trends["year_month"].astype(str)

# Plot traffic trends by category
fig = px.line(
    category_trends,
    x="year_month_str",
    y="people_in",
    color="bl1_label",
    title="Monthly Traffic Trends by Store Category (BL1)",
    labels={"people_in": "Total People In", "year_month_str": "Month"},
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Calculate growth rates for each category
def calculate_growth_rate(df, category_col):
    """Calculate monthly growth rate for each category."""
    results = []

    for category in df[category_col].dropna().unique():
        cat_data = df[df[category_col] == category].copy()
        cat_monthly = (
            cat_data.groupby("year_month").agg({"people_in": "sum"}).reset_index()
        )

        if len(cat_monthly) > 1:
            # Calculate percentage change
            cat_monthly["pct_change"] = cat_monthly["people_in"].pct_change() * 100

            # Calculate overall trend (linear regression slope)
            x = np.arange(len(cat_monthly))
            y = cat_monthly["people_in"].values

            if len(x) > 2:
                slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

                results.append(
                    {
                        "category": category,
                        "avg_monthly_change": cat_monthly["pct_change"].mean(),
                        "trend_slope": slope,
                        "r_squared": r_value**2,
                        "p_value": p_value,
                        "total_current": cat_monthly["people_in"].iloc[-1],
                        "total_start": cat_monthly["people_in"].iloc[0],
                    }
                )

    return pd.DataFrame(results)


# Calculate for BL1 categories
bl1_growth = calculate_growth_rate(stores_enriched, "bl1_label")
bl1_growth = bl1_growth.sort_values("trend_slope", ascending=False)

print("\nCategory Growth Analysis (BL1):")
print(bl1_growth)

In [ ]:
# Visualize category growth rates
fig = px.bar(
    bl1_growth.sort_values("avg_monthly_change", ascending=True),
    x="avg_monthly_change",
    y="category",
    title="Average Monthly Growth Rate by Category (BL1)",
    labels={"avg_monthly_change": "Avg Monthly % Change", "category": "Category"},
    orientation="h",
    color="avg_monthly_change",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# More detailed analysis for BL2 categories (top categories only)
top_bl2_categories = stores_enriched["bl2_label"].value_counts().head(15).index

bl2_growth = calculate_growth_rate(
    stores_enriched[stores_enriched["bl2_label"].isin(top_bl2_categories)], "bl2_label"
)
bl2_growth = bl2_growth.sort_values("trend_slope", ascending=False)

print("\nTop BL2 Categories - Growth Analysis:")
print(bl2_growth)

In [ ]:
# Visualize BL2 category trends
fig = px.bar(
    bl2_growth.sort_values("avg_monthly_change", ascending=True),
    x="avg_monthly_change",
    y="category",
    title="Average Monthly Growth Rate - Top 15 BL2 Categories",
    labels={"avg_monthly_change": "Avg Monthly % Change", "category": "Category"},
    orientation="h",
    color="avg_monthly_change",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(height=600)
fig.show()

## 5. Seasonality Analysis

In [ ]:
# Weekly patterns - which days are busiest?
day_of_week_traffic = (
    stores_enriched.groupby("day_name")
    .agg({"people_in": "mean", "store_average_dwell_time": "mean"})
    .reset_index()
)

# Order by day of week
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
day_of_week_traffic["day_name"] = pd.Categorical(
    day_of_week_traffic["day_name"], categories=day_order, ordered=True
)
day_of_week_traffic = day_of_week_traffic.sort_values("day_name")

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Average Traffic by Day of Week",
        "Average Dwell Time by Day of Week",
    ),
)

fig.add_trace(
    go.Bar(
        x=day_of_week_traffic["day_name"],
        y=day_of_week_traffic["people_in"],
        name="People In",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=day_of_week_traffic["day_name"],
        y=day_of_week_traffic["store_average_dwell_time"],
        name="Dwell Time",
    ),
    row=1,
    col=2,
)

fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# Monthly seasonality
month_traffic = (
    stores_enriched.groupby("month")
    .agg({"people_in": "mean", "store_average_dwell_time": "mean"})
    .reset_index()
)

month_names = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
month_traffic["month_name"] = month_traffic["month"].map(lambda x: month_names[x - 1])

fig = px.bar(
    month_traffic,
    x="month_name",
    y="people_in",
    title="Average Traffic by Month (Seasonality)",
    labels={"people_in": "Average People In", "month_name": "Month"},
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Perform seasonal decomposition on aggregate daily traffic
# Prepare time series data
ts_data = (
    daily_traffic.set_index("date")["people_in"].asfreq("D").fillna(method="ffill")
)

# Decompose if we have enough data
if len(ts_data) > 30:
    decomposition = seasonal_decompose(
        ts_data, model="additive", period=7
    )  # Weekly seasonality

    fig = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=("Original", "Trend", "Seasonal", "Residual"),
        vertical_spacing=0.08,
    )

    fig.add_trace(
        go.Scatter(
            x=decomposition.observed.index,
            y=decomposition.observed.values,
            mode="lines",
            name="Original",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=decomposition.trend.index,
            y=decomposition.trend.values,
            mode="lines",
            name="Trend",
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=decomposition.seasonal.index,
            y=decomposition.seasonal.values,
            mode="lines",
            name="Seasonal",
        ),
        row=3,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=decomposition.resid.index,
            y=decomposition.resid.values,
            mode="lines",
            name="Residual",
        ),
        row=4,
        col=1,
    )

    fig.update_layout(
        height=800,
        title_text="Time Series Decomposition (Weekly Seasonality)",
        showlegend=False,
    )
    fig.show()
else:
    print("Not enough data for seasonal decomposition")

## 6. Store Performance Metrics and Rankings

In [ ]:
# Merge with financial data
store_performance = stores_enriched.merge(
    store_financials, left_on="store_code", right_on="codstr", how="left"
)

# Convert financial columns to numeric (handle any non-numeric values)
store_performance["sales_r12m"] = pd.to_numeric(
    store_performance["sales_r12m"], errors="coerce"
)
store_performance["total_costs_r12m"] = pd.to_numeric(
    store_performance["total_costs_r12m"], errors="coerce"
)

# Calculate performance metrics per store
store_metrics = (
    store_performance.groupby(["store_code", "bl1_label", "bl2_label"])
    .agg(
        {
            "people_in": "sum",
            "people_window_flow": "sum",
            "store_average_dwell_time": "mean",
            "shopping_average_dwell_time": "mean",
            "average_visited_stores": "mean",
            "sales_r12m": "first",
            "total_costs_r12m": "first",
            "gla": "first",
        }
    )
    .reset_index()
)

# Calculate metrics
store_metrics["profit_r12m"] = (
    store_metrics["sales_r12m"] - store_metrics["total_costs_r12m"]
)
store_metrics["conversion_rate"] = (
    store_metrics["people_in"] / store_metrics["people_window_flow"] * 100
)
store_metrics["sales_per_sqm"] = store_metrics["sales_r12m"] / store_metrics["gla"]
store_metrics["traffic_per_sqm"] = store_metrics["people_in"] / store_metrics["gla"]

print("Store Performance Metrics:")
store_metrics.head(10)

In [ ]:
# Top performing stores by category
for category in store_metrics["bl1_label"].dropna().unique()[:5]:  # Top 5 categories
    print(f"\nTop stores in {category}:")
    cat_stores = store_metrics[store_metrics["bl1_label"] == category].nlargest(
        5, "sales_r12m"
    )
    print(
        cat_stores[
            ["store_code", "bl2_label", "sales_r12m", "people_in", "conversion_rate"]
        ]
    )

## 7. Trend Detection - Emerging vs Declining Categories

In [ ]:
# Identify emerging and declining trends
def classify_trend(row):
    """Classify trends based on growth rate (practical thresholds for retail)."""
    avg_change = row["avg_monthly_change"]

    # Use practical thresholds based on growth rate
    # (removed strict p-value requirement since 1 year of data has high variance)
    if avg_change > 2:  # Growing > 2% per month
        return "Strongly Emerging"
    elif avg_change > 0:
        return "Emerging"
    elif avg_change > -2:  # Small decline
        return "Stable"
    elif avg_change > -5:  # Moderate decline
        return "Declining"
    else:  # Declining > 5% per month
        return "Strongly Declining"


bl1_growth["trend_classification"] = bl1_growth.apply(classify_trend, axis=1)

print("\nTrend Classification Summary:")
print(
    bl1_growth[
        [
            "category",
            "avg_monthly_change",
            "trend_slope",
            "p_value",
            "trend_classification",
        ]
    ]
)

# Count by classification
print("\nDistribution of Trends:")
print(bl1_growth["trend_classification"].value_counts())

In [ ]:
# Visualize emerging vs declining categories
fig = px.scatter(
    bl1_growth,
    x="trend_slope",
    y="avg_monthly_change",
    size="total_current",
    color="trend_classification",
    hover_data=["category"],
    title="Category Trend Analysis: Emerging vs Declining",
    labels={"trend_slope": "Trend Slope", "avg_monthly_change": "Avg Monthly % Change"},
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=500)
fig.show()

In [ ]:
# Same analysis for BL2 categories
bl2_growth["trend_classification"] = bl2_growth.apply(classify_trend, axis=1)

print("\nBL2 Trend Classification:")
print(
    bl2_growth.sort_values("avg_monthly_change", ascending=False)[
        ["category", "avg_monthly_change", "trend_slope", "trend_classification"]
    ]
)

## 8. Time Series Forecasting

In [ ]:
# Note: Install Prophet if needed: pip install prophet
try:
    from prophet import Prophet

    prophet_available = True
except ImportError:
    print("Prophet not installed. Skipping forecasting section.")
    print("To install: pip install prophet")
    prophet_available = False

In [ ]:
if prophet_available:
    # Prepare data for Prophet (requires 'ds' and 'y' columns)
    forecast_data = daily_traffic[["date", "people_in"]].copy()
    forecast_data.columns = ["ds", "y"]
    forecast_data = forecast_data.dropna()

    # Initialize and fit the model
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
    )

    print("Fitting Prophet model...")
    model.fit(forecast_data)

    # Make future predictions (90 days ahead)
    future = model.make_future_dataframe(periods=90)
    forecast = model.predict(future)

    print("Forecast complete!")

In [ ]:
if prophet_available:
    # Visualize the forecast
    fig = go.Figure()

    # Historical data
    fig.add_trace(
        go.Scatter(
            x=forecast_data["ds"],
            y=forecast_data["y"],
            mode="markers",
            name="Historical",
            marker=dict(size=3, color="blue"),
        )
    )

    # Forecast
    fig.add_trace(
        go.Scatter(
            x=forecast["ds"],
            y=forecast["yhat"],
            mode="lines",
            name="Forecast",
            line=dict(color="red"),
        )
    )

    # Confidence interval
    fig.add_trace(
        go.Scatter(
            x=forecast["ds"],
            y=forecast["yhat_upper"],
            mode="lines",
            name="Upper Bound",
            line=dict(width=0),
            showlegend=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=forecast["ds"],
            y=forecast["yhat_lower"],
            mode="lines",
            name="Lower Bound",
            line=dict(width=0),
            fillcolor="rgba(255, 0, 0, 0.2)",
            fill="tonexty",
            showlegend=False,
        )
    )

    fig.update_layout(
        title="Traffic Forecast - Next 90 Days",
        xaxis_title="Date",
        yaxis_title="Total People In",
        height=500,
    )

    fig.show()

In [ ]:
if prophet_available:
    # Show components
    fig = model.plot_components(forecast)
    plt.tight_layout()
    plt.show()

In [ ]:
# Forecast by category
if prophet_available:
    category_forecasts = {}

    for category in (
        stores_enriched["bl1_label"].dropna().unique()[:3]
    ):  # Top 3 categories
        print(f"\nForecasting for {category}...")

        # Prepare category-specific data
        cat_data = (
            stores_enriched[stores_enriched["bl1_label"] == category]
            .groupby("date")
            .agg({"people_in": "sum"})
            .reset_index()
        )

        cat_data.columns = ["ds", "y"]
        cat_data = cat_data.dropna()

        if len(cat_data) > 30:  # Enough data to forecast
            # Fit model
            cat_model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=True,
                daily_seasonality=False,
            )
            cat_model.fit(cat_data)

            # Predict
            future = cat_model.make_future_dataframe(periods=90)
            cat_forecast = cat_model.predict(future)

            category_forecasts[category] = cat_forecast
            print(f"  Forecast complete for {category}")

    print(f"\nCreated forecasts for {len(category_forecasts)} categories")

In [ ]:
# Visualize category forecasts
if prophet_available and category_forecasts:
    fig = make_subplots(
        rows=len(category_forecasts),
        cols=1,
        subplot_titles=list(category_forecasts.keys()),
        vertical_spacing=0.1,
    )

    for i, (category, cat_forecast) in enumerate(category_forecasts.items(), 1):
        # Get historical data for this category
        cat_historical = (
            stores_enriched[stores_enriched["bl1_label"] == category]
            .groupby("date")
            .agg({"people_in": "sum"})
            .reset_index()
        )

        # Historical points
        fig.add_trace(
            go.Scatter(
                x=cat_historical["date"],
                y=cat_historical["people_in"],
                mode="markers",
                name=f"{category} (Historical)",
                marker=dict(size=3),
                showlegend=True,
            ),
            row=i,
            col=1,
        )

        # Forecast line
        fig.add_trace(
            go.Scatter(
                x=cat_forecast["ds"],
                y=cat_forecast["yhat"],
                mode="lines",
                name=f"{category} (Forecast)",
                line=dict(color="red"),
            ),
            row=i,
            col=1,
        )

        # Confidence interval
        fig.add_trace(
            go.Scatter(
                x=cat_forecast["ds"],
                y=cat_forecast["yhat_upper"],
                mode="lines",
                line=dict(width=0),
                showlegend=False,
            ),
            row=i,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=cat_forecast["ds"],
                y=cat_forecast["yhat_lower"],
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                fillcolor="rgba(255, 0, 0, 0.1)",
                showlegend=False,
            ),
            row=i,
            col=1,
        )

    fig.update_layout(
        height=300 * len(category_forecasts),
        title_text="90-Day Traffic Forecast by Category",
        showlegend=False,
    )
    fig.show()

## 9. Key Insights and Recommendations

In [ ]:
# Generate summary insights
date_min = stores_enriched["date"].min().strftime("%Y-%m-%d")
date_max = stores_enriched["date"].max().strftime("%Y-%m-%d")

insights = {
    "data_period": f"{date_min} to {date_max}",
    "total_stores": stores_enriched["store_code"].nunique(),
    "total_malls": stores_enriched["mall_id"].nunique(),
    "emerging_categories": bl1_growth[
        bl1_growth["trend_classification"].str.contains("Emerging", na=False)
    ]["category"].tolist(),
    "declining_categories": bl1_growth[
        bl1_growth["trend_classification"].str.contains("Declining", na=False)
    ]["category"].tolist(),
    "busiest_day": day_of_week_traffic.loc[
        day_of_week_traffic["people_in"].idxmax(), "day_name"
    ],
    "busiest_month": month_traffic.loc[
        month_traffic["people_in"].idxmax(), "month_name"
    ],
}

print("=" * 60)
print("KEY INSIGHTS SUMMARY")
print("=" * 60)
print(f"\nData Coverage: {insights['data_period']}")
print(f"Total Stores Analyzed: {insights['total_stores']}")
print(f"Total Malls: {insights['total_malls']}")
print(f"\nBusiest Day of Week: {insights['busiest_day']}")
print(f"Busiest Month: {insights['busiest_month']}")
print(f"\nEmerging Categories: {', '.join(insights['emerging_categories'])}")
print(f"\nDeclining Categories: {', '.join(insights['declining_categories'])}")
print("\n" + "=" * 60)

In [ ]:
# Recommendations based on trends
print("\nRECOMMENDATIONS FOR RETAIL MIX ADAPTATION:")
print("=" * 60)

# Recommend increasing presence of emerging categories
if insights["emerging_categories"]:
    print("\n1. INCREASE PRESENCE:")
    for cat in insights["emerging_categories"]:
        growth_rate = bl1_growth[bl1_growth["category"] == cat][
            "avg_monthly_change"
        ].values[0]
        print(f"   - {cat}: Growing at {growth_rate:.1f}% per month")

# Recommend reconsidering declining categories
if insights["declining_categories"]:
    print("\n2. RECONSIDER OR REPOSITION:")
    for cat in insights["declining_categories"]:
        growth_rate = bl1_growth[bl1_growth["category"] == cat][
            "avg_monthly_change"
        ].values[0]
        print(f"   - {cat}: Declining at {growth_rate:.1f}% per month")

print("\n3. OPTIMIZE FOR PEAK TIMES:")
print(f"   - Focus marketing efforts on {insights['busiest_day']}s")
print(f"   - Plan seasonal campaigns around {insights['busiest_month']}")

print("\n" + "=" * 60)